# 📧 Exploratory Data Analysis (EDA) & Data Preprocessing
# Dataset: Phishing Email Detection

---

## Dataset Information

| Attribute | Value |
|---|---|
| **Dataset Name** | Phishing Email Detection Dataset |
| **Description** | A labeled email corpus containing email body text classified as either Safe Email or Phishing Email, designed for binary text classification in the Cybersecurity domain. |
| **Format** | CSV |
| **Target Variable** | `Email Type` — 2 classes: `Safe Email` (0) and `Phishing Email` (1) |
| **Methodology** | CRISP-DM (Cross-Industry Standard Process for Data Mining) |

---

## Table of Contents

1. [Section 1: Initialization & Descriptive Statistics Overview](#section-1)
2. [Section 2: Univariate Analysis](#section-2)
3. [Section 3: Multivariate & Correlation Analysis](#section-3)
4. [Section 4: Data Cleaning & Preprocessing](#section-4)

---

> **Note:** Since this is an NLP dataset (text data), the EDA pipeline has been adapted accordingly:  
> - Extract numerical features (Feature Engineering) from text: email length, word count, special character count...  
> - Analyze word frequency, Word Cloud, and length distributions  
> - Apply NLP preprocessing: lowercasing, stopword removal, tokenization

<a id='section-1'></a>
# SECTION 1: Initialization & Descriptive Statistics Overview
## (Overview & Basic Inspection)

### 1.1 Import Required Libraries

In [ ]:
# ============================================================
# 1.1 — Import all required libraries
# ============================================================

# --- Data processing libraries ---
import pandas as pd                    # Tabular data processing (DataFrame)
import numpy as np                     # Numerical computation, arrays

# --- Visualization libraries ---
import matplotlib.pyplot as plt        # Basic plotting
import matplotlib.ticker as ticker     # Axis formatting
import seaborn as sns                  # Advanced statistical visualizations

# --- Text processing libraries (NLP) ---
import re                              # Regular expressions
import string                          # String constants
from collections import Counter        # Element frequency counting

# --- Machine Learning libraries ---
from sklearn.model_selection import train_test_split   # Train/Test splitting
from sklearn.preprocessing import LabelEncoder         # Label encoding
from sklearn.preprocessing import StandardScaler       # Feature scaling

# --- Display configuration ---
import warnings
warnings.filterwarnings('ignore')      # Suppress unnecessary warnings

# Configure pandas display options
pd.set_option('display.max_columns', None)       # Show all columns
pd.set_option('display.max_colwidth', 200)        # Max 200 chars per column
pd.set_option('display.float_format', '{:.4f}'.format)  # 4 decimal places

# Configure matplotlib for inline display in notebook
%matplotlib inline

# Set seaborn theme — modern, clean interface
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)          # Default figure size
plt.rcParams['figure.dpi'] = 100                   # Figure resolution
plt.rcParams['axes.titlesize'] = 14                # Title font size
plt.rcParams['axes.labelsize'] = 12                # Axis label font size

print("✅ Libraries imported successfully!")
print(f"   📦 pandas   : {pd.__version__}")
print(f"   📦 numpy    : {np.__version__}")
print(f"   📦 seaborn  : {sns.__version__}")

### 1.2 Load Data & Quick Inspection

In [ ]:
# ============================================================
# 1.2 — Load CSV file and quick data inspection
# ============================================================

# Path to the raw CSV file
DATA_PATH = '../data/raw/phishing-email-dataset.csv'

# Read the CSV file into a DataFrame
# - index_col=0: use the first column ('Unnamed: 0') as index instead of keeping it as a separate column
df_raw = pd.read_csv(DATA_PATH, index_col=0)

# Create a working copy, preserving the original for reference
df = df_raw.copy()

print(f"📂 Dataset loaded successfully from: {DATA_PATH}")
print(f"📊 Dataset dimensions: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"💾 Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

In [ ]:
# --- Display the first 5 rows ---
print("="*80)
print("📋 FIRST 5 ROWS (head)")
print("="*80)
display(df.head())

In [ ]:
# --- Display the last 5 rows ---
print("="*80)
print("📋 LAST 5 ROWS (tail)")
print("="*80)
display(df.tail())

### 1.3 Data Types & Structure Inspection

In [ ]:
# ============================================================
# 1.3 — Detailed data type & structure inspection
# ============================================================

print("="*80)
print("🔍 DATASET STRUCTURE INFORMATION (info)")
print("="*80)
df.info()

print("\n" + "="*80)
print("📊 DATA TYPES PER COLUMN")
print("="*80)

# Create a data type summary table
dtype_summary = pd.DataFrame({
    'Column Name': df.columns,
    'Data Type': df.dtypes.values,
    'Unique Values': df.nunique().values,
    'Sample Value': [df[col].iloc[0] if len(df) > 0 else 'N/A' for col in df.columns]
})
display(dtype_summary)

### 1.4 Missing Values Analysis

In [ ]:
# ============================================================
# 1.4 — Detailed missing values statistics
# ============================================================

def analyze_missing_values(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze and create a missing values summary table.
    
    Parameters:
        dataframe: DataFrame to analyze
    Returns:
        DataFrame containing missing values info, sorted by percentage descending
    """
    # Count null values in each column
    missing_count = dataframe.isnull().sum()
    
    # Calculate missing percentage
    missing_pct = (missing_count / len(dataframe)) * 100
    
    # Create summary table
    missing_table = pd.DataFrame({
        'Column Name': missing_count.index,
        'Missing Count': missing_count.values,
        'Missing %': missing_pct.values,
        'Data Type': dataframe.dtypes.values
    })
    
    # Sort by missing percentage descending
    missing_table = missing_table.sort_values('Missing %', ascending=False)
    missing_table = missing_table.reset_index(drop=True)
    
    return missing_table

# Call the function and display results
missing_df = analyze_missing_values(df)

print("="*80)
print("❓ MISSING VALUES SUMMARY TABLE")
print("="*80)
display(missing_df)

# Quick summary
total_missing = df.isnull().sum().sum()
total_cells = df.shape[0] * df.shape[1]
print(f"\n📊 Total data cells      : {total_cells:,}")
print(f"❌ Total missing cells   : {total_missing:,}")
print(f"📉 Overall missing rate  : {(total_missing/total_cells)*100:.4f}%")

In [ ]:
# --- Visualize missing data with a bar chart ---

# Only plot if there are columns with missing values
cols_with_missing = missing_df[missing_df['Missing Count'] > 0]

if len(cols_with_missing) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    bars = ax.bar(
        cols_with_missing['Column Name'],
        cols_with_missing['Missing %'],
        color=sns.color_palette('Reds_r', len(cols_with_missing)),
        edgecolor='black',
        linewidth=0.5
    )
    
    # Add value labels on top of each bar
    for bar, count in zip(bars, cols_with_missing['Missing Count']):
        ax.text(
            bar.get_x() + bar.get_width() / 2.,
            bar.get_height() + 0.3,
            f'{count:,}\n({bar.get_height():.2f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold'
        )
    
    ax.set_title('📊 Missing Data Percentage by Column', fontsize=14, fontweight='bold')
    ax.set_ylabel('Missing Rate (%)')
    ax.set_xlabel('Column Name')
    plt.tight_layout()
    plt.show()
else:
    print("✅ No columns have missing data!")

### 1.5 Duplicate Data Check

In [ ]:
# ============================================================
# 1.5 — Check and report duplicate data
# ============================================================

# Count fully duplicate rows (all columns identical)
num_duplicates = df.duplicated().sum()
pct_duplicates = (num_duplicates / len(df)) * 100

print("="*80)
print("🔄 DUPLICATE DATA CHECK")
print("="*80)
print(f"   Total rows           : {len(df):,}")
print(f"   Duplicate rows       : {num_duplicates:,}")
print(f"   Duplicate rate       : {pct_duplicates:.2f}%")

# Display sample duplicate rows (if any)
if num_duplicates > 0:
    print(f"\n📋 Sample of 5 duplicate rows:")
    display(df[df.duplicated(keep='first')].head())

### 1.6 Descriptive Statistics

In [ ]:
# ============================================================
# 1.6 — Descriptive statistics for categorical variables
# ============================================================

print("="*80)
print("📊 DESCRIPTIVE STATISTICS — CATEGORICAL VARIABLES")
print("="*80)

# Use include='O' (Object) to only describe string-type variables
display(df.describe(include='O'))

### 1.7 Feature Engineering — Extract Numerical Features from Text

> Since the dataset is primarily text data, we need to extract numerical features from the email content for statistical analysis and visualization.

In [ ]:
# ============================================================
# 1.7 — Extract numerical features from the "Email Text" column
# ============================================================

def extract_text_features(dataframe: pd.DataFrame, text_col: str = 'Email Text') -> pd.DataFrame:
    """
    Extract statistical features from the text column.
    
    Parameters:
        dataframe: Source DataFrame
        text_col: Name of the column containing text content
    Returns:
        DataFrame with new feature columns added
    """
    df_feat = dataframe.copy()
    
    # Fill NaN values with empty strings to avoid errors during string processing
    text_series = df_feat[text_col].fillna('')
    
    # 1. Email length (character count)
    df_feat['char_count'] = text_series.str.len()
    
    # 2. Word count in the email
    df_feat['word_count'] = text_series.str.split().str.len().fillna(0).astype(int)
    
    # 3. Average word length
    df_feat['avg_word_length'] = df_feat.apply(
        lambda row: np.mean([len(w) for w in str(row[text_col]).split()]) 
        if row['word_count'] > 0 else 0,
        axis=1
    )
    
    # 4. Sentence count (estimated by punctuation marks)
    df_feat['sentence_count'] = text_series.str.count(r'[.!?]+').clip(lower=1)
    
    # 5. Uppercase character count
    df_feat['uppercase_count'] = text_series.str.count(r'[A-Z]')
    
    # 6. Uppercase character ratio / total characters
    df_feat['uppercase_ratio'] = (
        df_feat['uppercase_count'] / df_feat['char_count'].replace(0, 1)
    )
    
    # 7. Special character count (non-alphanumeric)
    df_feat['special_char_count'] = text_series.str.count(r'[^a-zA-Z0-9\s]')
    
    # 8. URL count in the email
    df_feat['url_count'] = text_series.str.count(
        r'https?://[^\s]+|www\.[^\s]+'
    )
    
    # 9. Email address count in the content
    df_feat['email_addr_count'] = text_series.str.count(
        r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    )
    
    # 10. Digit count
    df_feat['digit_count'] = text_series.str.count(r'\d')
    
    return df_feat

# Apply the feature extraction function
df = extract_text_features(df)

print("✅ Successfully extracted 10 numerical features from email text!")
print(f"📊 New dataset dimensions: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\n📋 Current column list:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2d}. {col}")

In [ ]:
# --- Descriptive statistics for the newly extracted numerical features ---

print("="*80)
print("📊 DESCRIPTIVE STATISTICS — NUMERICAL FEATURES")
print("="*80)

# Get the list of numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"   Number of numeric features: {len(numeric_cols)}")
print(f"   Feature list: {numeric_cols}\n")

# Display describe() with transpose for readability
display(df[numeric_cols].describe().T)

---

**Data Scientist's Notes:**

> _[Analysis notes after running Section 1 — e.g.: Dataset size, missing rate, whether data types are reasonable, whether any type conversions are needed, preliminary observations about feature distributions...]_

---

<a id='section-2'></a>
# SECTION 2: Univariate Analysis

### 2.1 Target Variable Analysis — `Email Type`

> Check the distribution between 2 classes: **Safe Email** vs **Phishing Email**  
> → Evaluate the degree of **Class Imbalance**

In [ ]:
# ============================================================
# 2.1 — Target variable analysis: Email Type
# ============================================================

TARGET_COL = 'Email Type'  # Target column name

# Count frequency of each class
target_counts = df[TARGET_COL].value_counts()
target_pct = df[TARGET_COL].value_counts(normalize=True) * 100

print("="*80)
print("🎯 TARGET VARIABLE DISTRIBUTION — Email Type")
print("="*80)

# Create summary table
target_summary = pd.DataFrame({
    'Class': target_counts.index,
    'Count': target_counts.values,
    'Percentage (%)': target_pct.values
})
display(target_summary)

# Calculate imbalance ratio
imbalance_ratio = target_counts.max() / target_counts.min()
print(f"\n⚖️  Imbalance Ratio: {imbalance_ratio:.2f}:1")

if imbalance_ratio > 3:
    print("⚠️  WARNING: Dataset is SEVERELY imbalanced! Consider techniques like SMOTE, class_weight, etc.")
elif imbalance_ratio > 1.5:
    print("🔶 Dataset has mild imbalance. Monitor performance on the minority class.")
else:
    print("✅ Dataset is relatively balanced.")

In [ ]:
# --- Visualize Target distribution ---

# Color palette for each class
TARGET_COLORS = {'Safe Email': '#2ecc71', 'Phishing Email': '#e74c3c'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Countplot (bar chart) ---
ax1 = axes[0]
bars = ax1.bar(
    target_counts.index,
    target_counts.values,
    color=[TARGET_COLORS[label] for label in target_counts.index],
    edgecolor='black',
    linewidth=0.8,
    width=0.5
)

# Add value labels
for bar, count, pct in zip(bars, target_counts.values, target_pct.values):
    ax1.text(
        bar.get_x() + bar.get_width() / 2.,
        bar.get_height() + len(df) * 0.01,
        f'{count:,}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax1.set_title('📊 Target Variable Distribution (Countplot)', fontweight='bold')
ax1.set_ylabel('Number of Emails')
ax1.set_xlabel('Email Type')

# --- Plot 2: Pie Chart ---
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    target_counts.values,
    labels=target_counts.index,
    autopct='%1.1f%%',
    colors=[TARGET_COLORS[label] for label in target_counts.index],
    startangle=90,
    explode=[0.03] * len(target_counts),
    shadow=True,
    textprops={'fontsize': 12}
)
# Bold the percentage text
for autotext in autotexts:
    autotext.set_fontweight('bold')

ax2.set_title('🥧 Class Proportions (Pie Chart)', fontweight='bold')

plt.suptitle('TARGET VARIABLE ANALYSIS — Email Type', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.2 Numerical Feature Distribution Analysis

> Automatically plot **Histogram + KDE** and **Boxplot** for all numerical features to detect:  
> - Distribution skewness  
> - Outliers

In [ ]:
# ============================================================
# 2.2 — Plot Histogram + KDE + Boxplot for numerical features
# ============================================================

def plot_numeric_distributions(dataframe: pd.DataFrame, 
                                numeric_columns: list,
                                cols_per_row: int = 2) -> None:
    """
    Automatically plot paired charts (Histogram+KDE, Boxplot) for each numerical feature.
    
    Parameters:
        dataframe: DataFrame containing the data
        numeric_columns: List of numeric column names to plot
        cols_per_row: Number of chart pairs per row
    """
    n_features = len(numeric_columns)
    
    # Each feature occupies 1 row with 2 columns (Histogram | Boxplot)
    fig, axes = plt.subplots(
        nrows=n_features, ncols=2,
        figsize=(16, 4 * n_features)
    )
    
    # Ensure axes is always a 2D array
    if n_features == 1:
        axes = axes.reshape(1, -1)
    
    # Diverse color palette
    colors = sns.color_palette('husl', n_features)
    
    for i, col in enumerate(numeric_columns):
        # --- Histogram + KDE (left column) ---
        ax_hist = axes[i, 0]
        sns.histplot(
            data=dataframe, x=col, kde=True,
            color=colors[i], ax=ax_hist,
            bins=50, edgecolor='white', linewidth=0.5
        )
        
        # Add skewness & kurtosis information
        skew_val = dataframe[col].skew()
        kurt_val = dataframe[col].kurtosis()
        ax_hist.axvline(dataframe[col].median(), color='red', linestyle='--', 
                        linewidth=1.5, label=f'Median: {dataframe[col].median():.2f}')
        ax_hist.axvline(dataframe[col].mean(), color='blue', linestyle='-.',
                        linewidth=1.5, label=f'Mean: {dataframe[col].mean():.2f}')
        ax_hist.set_title(f'📊 Histogram + KDE: {col}\n(Skew={skew_val:.2f}, Kurt={kurt_val:.2f})',
                         fontsize=11, fontweight='bold')
        ax_hist.legend(fontsize=9)
        
        # --- Boxplot (right column) ---
        ax_box = axes[i, 1]
        sns.boxplot(
            data=dataframe, x=col,
            color=colors[i], ax=ax_box,
            flierprops=dict(marker='o', markersize=3, alpha=0.5)
        )
        
        # Count outliers using IQR
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1
        n_outliers = ((dataframe[col] < Q1 - 1.5 * IQR) | (dataframe[col] > Q3 + 1.5 * IQR)).sum()
        pct_outliers = (n_outliers / len(dataframe)) * 100
        
        ax_box.set_title(
            f'📦 Boxplot: {col}\n(Outliers: {n_outliers:,} — {pct_outliers:.2f}%)',
            fontsize=11, fontweight='bold'
        )
    
    plt.suptitle(
        'NUMERICAL FEATURE DISTRIBUTIONS',
        fontsize=16, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()

# Call the plotting function for all numeric features
plot_numeric_distributions(df, numeric_cols)

In [ ]:
# --- Skewness and Outlier summary table for all numeric features ---

def summarize_numeric_features(dataframe: pd.DataFrame, 
                                numeric_columns: list) -> pd.DataFrame:
    """
    Create a summary table of skewness, kurtosis, and outlier count (IQR method).
    """
    summary_rows = []
    
    for col in numeric_columns:
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        n_outliers = ((dataframe[col] < lower_bound) | (dataframe[col] > upper_bound)).sum()
        
        summary_rows.append({
            'Feature': col,
            'Mean': dataframe[col].mean(),
            'Median': dataframe[col].median(),
            'Std': dataframe[col].std(),
            'Skewness': dataframe[col].skew(),
            'Kurtosis': dataframe[col].kurtosis(),
            'IQR': IQR,
            'Outliers': n_outliers,
            'Outlier %': (n_outliers / len(dataframe)) * 100
        })
    
    return pd.DataFrame(summary_rows)

outlier_summary = summarize_numeric_features(df, numeric_cols)

print("="*80)
print("📊 SUMMARY TABLE: SKEWNESS & OUTLIERS (IQR Method)")
print("="*80)
display(outlier_summary)

### 2.3 Distribution Analysis by Target — Safe vs Phishing Comparison

> Compare feature distributions between the 2 email classes to find strong discriminating features.

In [ ]:
# ============================================================
# 2.3 — KDE distribution split by Target for each numeric feature
# ============================================================

def plot_distributions_by_target(dataframe: pd.DataFrame,
                                  numeric_columns: list,
                                  target_col: str = 'Email Type',
                                  palette: dict = None) -> None:
    """
    Plot KDE plots comparing distributions across target variable classes.
    """
    if palette is None:
        palette = {'Safe Email': '#2ecc71', 'Phishing Email': '#e74c3c'}
    
    n_features = len(numeric_columns)
    n_cols = 3  # 3 plots per row
    n_rows = (n_features + n_cols - 1) // n_cols  # Round up
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
    axes = axes.flatten()  # Flatten for easy iteration
    
    for i, col in enumerate(numeric_columns):
        ax = axes[i]
        
        # Plot KDE for each class
        for label, color in palette.items():
            subset = dataframe[dataframe[target_col] == label][col].dropna()
            if len(subset) > 0:
                sns.kdeplot(
                    data=subset, ax=ax,
                    color=color, fill=True, alpha=0.3,
                    label=label, linewidth=2
                )
        
        ax.set_title(f'{col}', fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
    
    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle(
        '📊 FEATURE DISTRIBUTIONS BY EMAIL TYPE (Safe vs Phishing)',
        fontsize=16, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()

# Call the plotting function
plot_distributions_by_target(df, numeric_cols, TARGET_COL, TARGET_COLORS)

### 2.4 Email Length Analysis — Boxplot by Target

In [ ]:
# ============================================================
# 2.4 — Boxplot comparing key features by Target
# ============================================================

# Select 4 most important features for detailed comparison
key_features = ['char_count', 'word_count', 'uppercase_ratio', 'url_count']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(key_features):
    ax = axes[i]
    sns.boxplot(
        data=df, x=TARGET_COL, y=col,
        palette=TARGET_COLORS, ax=ax,
        flierprops=dict(marker='o', markersize=2, alpha=0.3)
    )
    ax.set_title(f'📦 {col} by Email Type', fontweight='bold')
    ax.set_xlabel('')

plt.suptitle(
    'KEY FEATURE COMPARISON: SAFE vs PHISHING',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

### 2.5 Top Word Frequency Analysis

In [ ]:
# ============================================================
# 2.5 — Word frequency analysis for each email type
# ============================================================

# Basic English stopword list
STOP_WORDS = set([
    'the', 'a', 'an', 'is', 'it', 'to', 'in', 'of', 'and', 'for',
    'on', 'that', 'this', 'with', 'you', 'are', 'was', 'be', 'as',
    'at', 'by', 'we', 'or', 'not', 'from', 'but', 'have', 'has',
    'had', 'do', 'does', 'did', 'will', 'would', 'can', 'could',
    'should', 'may', 'might', 'shall', 'i', 'he', 'she', 'they',
    'me', 'him', 'her', 'us', 'them', 'my', 'your', 'his', 'its',
    'our', 'their', 'what', 'which', 'who', 'whom', 'if', 'then',
    'than', 'so', 'no', 'all', 'each', 'any', 'both', 'more',
    'most', 'other', 'some', 'such', 'only', 'own', 'same', 'too',
    'very', 'just', 'because', 'how', 'when', 'where', 'there',
    'here', 'about', 'up', 'out', 'been', 'being', 'these', 'those',
    'also', 'after', 'before', 'between', 'through', 'during',
    'above', 'below', 'into', 'over', 'under', 'again', 'further',
    'once', 'am', 'were', 'don', 't', 's', 're', 've', 'll', 'd', 'm'
])


def get_top_words(texts: pd.Series, n_top: int = 20, 
                  stop_words: set = STOP_WORDS) -> list:
    """
    Extract top N most frequent words (after stopword removal).
    
    Parameters:
        texts: Series containing text content
        n_top: Number of top words to return
        stop_words: Set of stopwords to remove
    Returns:
        List of tuples (word, frequency)
    """
    # Concatenate all text and convert to lowercase
    all_text = ' '.join(texts.dropna().astype(str)).lower()
    
    # Keep only letters and whitespace
    all_text = re.sub(r'[^a-z\s]', '', all_text)
    
    # Tokenize and remove stopwords + short words
    words = [w for w in all_text.split() if w not in stop_words and len(w) > 2]
    
    # Count frequency and return top N
    return Counter(words).most_common(n_top)


# Get top words for each email type
safe_texts = df[df[TARGET_COL] == 'Safe Email']['Email Text']
phishing_texts = df[df[TARGET_COL] == 'Phishing Email']['Email Text']

top_safe = get_top_words(safe_texts, n_top=20)
top_phishing = get_top_words(phishing_texts, n_top=20)

# --- Plot word frequency charts ---
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Safe Email
words_safe, counts_safe = zip(*top_safe)
axes[0].barh(words_safe[::-1], counts_safe[::-1], color='#2ecc71', edgecolor='black', linewidth=0.5)
axes[0].set_title('✅ Top 20 Words — Safe Email', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Frequency')

# Phishing Email
words_phish, counts_phish = zip(*top_phishing)
axes[1].barh(words_phish[::-1], counts_phish[::-1], color='#e74c3c', edgecolor='black', linewidth=0.5)
axes[1].set_title('🚨 Top 20 Words — Phishing Email', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Frequency')

plt.suptitle('WORD FREQUENCY ANALYSIS BY EMAIL TYPE', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

**Data Scientist's Notes:**

> _[Analysis notes after running Section 2 — e.g.: Balance ratio between 2 classes, which distributions are heavily skewed, which features have many outliers, discriminating keywords between Safe and Phishing emails, ...]_

---

<a id='section-3'></a>
# SECTION 3: Multivariate & Correlation Analysis

### 3.1 Correlation Matrix Heatmap

> Detect **Multicollinearity** between numerical features.

In [ ]:
# ============================================================
# 3.1 — Correlation matrix between numerical features
# ============================================================

# Compute Pearson correlation matrix
corr_matrix = df[numeric_cols].corr()

# Create mask for the upper triangle (remove duplicates)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 10))

sns.heatmap(
    corr_matrix,
    mask=mask,                    # Display only lower triangle
    annot=True,                   # Show correlation values
    fmt='.2f',                    # Format to 2 decimal places
    cmap='RdBu_r',               # Color map: red (negative) → white (0) → blue (positive)
    vmin=-1, vmax=1,             # Color scale limits [-1, 1]
    center=0,                     # Center color at 0
    square=True,                  # Square cells for aesthetics
    linewidths=0.5,              # Border width between cells
    cbar_kws={'label': 'Pearson Correlation Coefficient'},
    ax=ax
)

ax.set_title(
    '🔥 CORRELATION MATRIX HEATMAP\nDetecting Multicollinearity Between Features',
    fontsize=14, fontweight='bold', pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
# --- List feature pairs with HIGH correlation (> 0.7 or < -0.7) ---

def find_high_correlations(corr_matrix: pd.DataFrame, threshold: float = 0.7) -> pd.DataFrame:
    """
    Find feature pairs with correlation exceeding the threshold.
    
    Parameters:
        corr_matrix: Correlation matrix
        threshold: Correlation threshold (default 0.7)
    Returns:
        DataFrame containing highly correlated pairs
    """
    pairs = []
    
    # Iterate through the lower triangle of the matrix (avoid duplicates)
    for i in range(len(corr_matrix.columns)):
        for j in range(i + 1, len(corr_matrix.columns)):
            corr_val = corr_matrix.iloc[i, j]
            if abs(corr_val) >= threshold:
                pairs.append({
                    'Feature 1': corr_matrix.columns[i],
                    'Feature 2': corr_matrix.columns[j],
                    'Correlation': corr_val,
                    'Abs Correlation': abs(corr_val)
                })
    
    result = pd.DataFrame(pairs).sort_values('Abs Correlation', ascending=False)
    return result.reset_index(drop=True)

high_corr = find_high_correlations(corr_matrix, threshold=0.7)

print("="*80)
print(f"⚠️  FEATURE PAIRS WITH HIGH CORRELATION (|r| ≥ 0.7)")
print("="*80)

if len(high_corr) > 0:
    display(high_corr)
    print("\n💡 Suggestion: Consider removing 1 of the 2 features in each pair to avoid multicollinearity.")
else:
    print("✅ No feature pairs exceed the 0.7 correlation threshold.")

### 3.2 Feature-Target Correlation (Target Correlation)

In [ ]:
# ============================================================
# 3.2 — Correlation between features and target variable
# ============================================================

# Encode target variable as numeric for correlation computation
# Safe Email = 0, Phishing Email = 1
target_encoded = (df[TARGET_COL] == 'Phishing Email').astype(int)

# Compute Point-Biserial correlation between each numeric feature and Target
target_corr = df[numeric_cols].corrwith(target_encoded).sort_values(ascending=True)

# Plot horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in target_corr.values]

bars = ax.barh(
    target_corr.index,
    target_corr.values,
    color=colors,
    edgecolor='black',
    linewidth=0.5
)

# Add value labels
for bar, val in zip(bars, target_corr.values):
    ax.text(
        val + (0.005 if val > 0 else -0.005),
        bar.get_y() + bar.get_height() / 2.,
        f'{val:.3f}',
        ha='left' if val > 0 else 'right',
        va='center', fontsize=10, fontweight='bold'
    )

# Reference line at 0
ax.axvline(x=0, color='black', linewidth=0.8)

ax.set_title(
    '🎯 FEATURE-TARGET CORRELATION\n'
    '(Red = Positive correlation with Phishing | Green = Negative correlation)',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Correlation Coefficient (Point-Biserial)')

plt.tight_layout()
plt.show()

### 3.3 Pairplot — Most Important Features

In [ ]:
# ============================================================
# 3.3 — Pairplot for top 4 features most correlated with Target
# ============================================================

# Select top 4 features with highest absolute correlation with Target
top_features = target_corr.abs().nlargest(4).index.tolist()

print(f"📊 Top 4 features most correlated with Target: {top_features}")

# Take a random subsample for Pairplot (full dataset would be very slow)
SAMPLE_SIZE = 5000  # Random sample of 5000
df_sample = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42)

# Plot Pairplot
pair_plot = sns.pairplot(
    df_sample,
    vars=top_features,
    hue=TARGET_COL,
    palette=TARGET_COLORS,
    diag_kind='kde',        # Diagonal uses KDE instead of histogram
    plot_kws={'alpha': 0.4, 's': 15},  # Transparent, small points
    diag_kws={'fill': True, 'alpha': 0.5}
)

pair_plot.fig.suptitle(
    'PAIRPLOT — Top Features Most Correlated With Target',
    fontsize=14, fontweight='bold', y=1.02
)
plt.show()

### 3.4 Feature Mean Comparison by Target

In [ ]:
# ============================================================
# 3.4 — Compare mean feature values between the 2 classes
# ============================================================

# Compute mean and median by Target group
group_stats = df.groupby(TARGET_COL)[numeric_cols].agg(['mean', 'median'])

print("="*80)
print("📊 FEATURE MEAN & MEDIAN COMPARISON BY EMAIL TYPE")
print("="*80)
display(group_stats)

---

**Data Scientist's Notes:**

> _[Analysis notes after running Section 3 — e.g.: Which feature pairs have high multicollinearity, which feature discriminates most between Safe and Phishing, whether to remove any features, interesting patterns from pairplot, ...]_

---

<a id='section-4'></a>
# SECTION 4: Data Cleaning & Preprocessing

### 4.1 Missing Values Treatment

In [ ]:
# ============================================================
# 4.1 — Handle missing values
# ============================================================

# Create a working copy, preserving df for reference
df_clean = df.copy()

print("="*80)
print("🧹 MISSING VALUES TREATMENT")
print("="*80)

# --- Check missing values before treatment ---
print("\n📋 BEFORE TREATMENT:")
print(df_clean.isnull().sum())

# --- Treatment strategy ---
# Column 'Email Text': Fill with empty string if missing (primary text column)
# Numeric columns: Fill with median (more robust than mean for skewed distributions)

# Handle text column: fill with empty string
if df_clean['Email Text'].isnull().sum() > 0:
    n_missing_text = df_clean['Email Text'].isnull().sum()
    df_clean['Email Text'] = df_clean['Email Text'].fillna('')
    print(f"\n   ✅ 'Email Text': Filled {n_missing_text} NaN values with empty string.")

# Handle target column: drop rows if missing (cannot infer labels)
if df_clean[TARGET_COL].isnull().sum() > 0:
    n_before = len(df_clean)
    df_clean = df_clean.dropna(subset=[TARGET_COL])
    n_dropped = n_before - len(df_clean)
    print(f"   ❌ '{TARGET_COL}': Dropped {n_dropped} rows with missing target labels.")

# Handle numeric columns: fill with median
numeric_cols_current = df_clean.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols_current:
    n_missing = df_clean[col].isnull().sum()
    if n_missing > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f"   ✅ '{col}': Filled {n_missing} NaN values with median ({median_val:.2f}).")

# --- Check after treatment ---
print("\n📋 AFTER TREATMENT:")
print(df_clean.isnull().sum())
print(f"\n📊 Dataset size after treatment: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

### 4.2 Duplicate Removal

In [ ]:
# ============================================================
# 4.2 — Handle duplicate data
# ============================================================

print("="*80)
print("🔄 DUPLICATE DATA REMOVAL")
print("="*80)

n_before = len(df_clean)
n_dups = df_clean.duplicated().sum()

print(f"   Rows before treatment   : {n_before:,}")
print(f"   Duplicates detected     : {n_dups:,}")

if n_dups > 0:
    # Remove duplicate rows, keeping the first occurrence
    df_clean = df_clean.drop_duplicates(keep='first').reset_index(drop=True)
    print(f"   ✅ Removed {n_dups:,} duplicate rows.")
    print(f"   Rows remaining          : {len(df_clean):,}")
else:
    print("   ✅ No duplicate rows found!")

### 4.3 Outlier Detection and Treatment

In [ ]:
# ============================================================
# 4.3 — Detect and handle Outliers using the IQR method
# ============================================================

def detect_and_cap_outliers_iqr(dataframe: pd.DataFrame, 
                                 columns: list,
                                 multiplier: float = 1.5,
                                 method: str = 'cap') -> pd.DataFrame:
    """
    Detect and handle outliers using the IQR method.
    
    Parameters:
        dataframe: DataFrame to process
        columns: List of numeric columns to check for outliers
        multiplier: IQR multiplier (default 1.5)
        method: 'cap' = clip/winsorize, 'remove' = drop rows
    Returns:
        Processed DataFrame and outlier report DataFrame
    """
    df_result = dataframe.copy()
    outlier_report = []
    
    for col in columns:
        Q1 = df_result[col].quantile(0.25)
        Q3 = df_result[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR
        
        # Count outliers
        n_lower = (df_result[col] < lower_bound).sum()
        n_upper = (df_result[col] > upper_bound).sum()
        n_total = n_lower + n_upper
        
        if n_total > 0:
            if method == 'cap':
                # Clip/Winsorize: replace outlier values with IQR bounds
                df_result[col] = df_result[col].clip(lower=lower_bound, upper=upper_bound)
            
            outlier_report.append({
                'Feature': col,
                'Lower Bound': lower_bound,
                'Upper Bound': upper_bound,
                'Outliers (below)': n_lower,
                'Outliers (above)': n_upper,
                'Total Outliers': n_total,
                'Method': method.upper()
            })
    
    report_df = pd.DataFrame(outlier_report)
    return df_result, report_df


# Apply: Cap outliers for numeric columns
# Note: Only handle outliers on feature columns, NOT on the target column
numeric_features = df_clean.select_dtypes(include=[np.number]).columns.tolist()

df_clean, outlier_report = detect_and_cap_outliers_iqr(
    df_clean, 
    columns=numeric_features,
    multiplier=1.5,
    method='cap'  # Use clipping/winsorize method
)

print("="*80)
print("📊 OUTLIER TREATMENT REPORT (IQR × 1.5, Method: CAP/WINSORIZE)")
print("="*80)

if len(outlier_report) > 0:
    display(outlier_report)
    print("\n✅ All outliers have been clipped (winsorized) to IQR bounds.")
else:
    print("✅ No significant outliers detected.")

### 4.4 Categorical Encoding

> Encode the target variable `Email Type` as binary values:  
> - `Safe Email` → **0**  
> - `Phishing Email` → **1**

In [ ]:
# ============================================================
# 4.4 — Target variable encoding (Label Encoding)
# ============================================================

print("="*80)
print("🏷️  TARGET VARIABLE ENCODING (Label Encoding)")
print("="*80)

# Use sklearn's LabelEncoder
le = LabelEncoder()

# Fit & Transform: Encode Email Type
df_clean['label'] = le.fit_transform(df_clean[TARGET_COL])

# Display the mapping
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"   📋 Mapping: {label_mapping}")

# Verify the result
print(f"\n   📊 Distribution after encoding:")
print(df_clean['label'].value_counts().to_string())

# Display sample
print("\n   📋 Sample after encoding:")
display(df_clean[[TARGET_COL, 'label']].drop_duplicates())

### 4.5 Feature Preparation & Train/Test Split

In [ ]:
# ============================================================
# 4.5 — Prepare Feature Matrix and Train/Test Split
# ============================================================

print("="*80)
print("✂️  TRAIN / TEST SPLIT")
print("="*80)

# --- Define Feature Matrix (X) and Target Vector (y) ---

# Numeric feature columns (exclude original text column, original target column, and label column)
feature_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in feature_cols if c != 'label']  # Remove label column

X = df_clean[feature_cols]  # Feature matrix
y = df_clean['label']       # Target vector

print(f"   📊 Feature Matrix (X) : {X.shape}")
print(f"   🎯 Target Vector  (y) : {y.shape}")
print(f"   📋 Features           : {feature_cols}")

# --- Split Train/Test at 80/20 ratio ---
# - stratify=y: Ensure class ratio is preserved in both Train and Test
# - random_state=42: Ensure reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% for Test
    random_state=42,      # Fixed seed for reproducibility
    stratify=y            # Stratified by label
)

print(f"\n   📦 Train set : {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   📦 Test set  : {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

# Check label distribution in Train and Test
print("\n   ⚖️  Label distribution:")
train_dist = y_train.value_counts(normalize=True) * 100
test_dist = y_test.value_counts(normalize=True) * 100

dist_check = pd.DataFrame({
    'Train (%)': train_dist,
    'Test (%)': test_dist,
    'Original (%)': y.value_counts(normalize=True) * 100
})
display(dist_check)

### 4.6 Feature Scaling

> **IMPORTANT**: Only `fit` the scaler on the **Train** set, then `transform` both Train and Test.  
> → Prevent **Data Leakage** from the Test set into the preprocessing pipeline.

In [ ]:
# ============================================================
# 4.6 — Feature Scaling — StandardScaler
# ============================================================

print("="*80)
print("📏 FEATURE SCALING (StandardScaler)")
print("="*80)

# Initialize StandardScaler
# StandardScaler: z = (x - mean) / std → standard normal distribution (mean=0, std=1)
scaler = StandardScaler()

# ⚠️ IMPORTANT: Only FIT on TRAIN set to prevent Data Leakage
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),     # Fit + Transform on Train
    columns=feature_cols,
    index=X_train.index
)

# Transform Test set using the scaler fitted on Train
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),           # Only Transform (NO fit) on Test
    columns=feature_cols,
    index=X_test.index
)

print("   ✅ StandardScaler applied successfully!")
print("   ⚠️  Scaler fitted on TRAIN → transformed TEST (no data leakage).")

# --- Verify scaling results ---
print("\n📊 TRAIN set statistics after scaling:")
display(X_train_scaled.describe().T[['mean', 'std', 'min', 'max']])

print("\n📊 TEST set statistics after scaling:")
display(X_test_scaled.describe().T[['mean', 'std', 'min', 'max']])

### 4.7 Export Processed Data

In [ ]:
# ============================================================
# 4.7 — Save preprocessed data to files
# ============================================================

import os

# Directory for processed data
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)  # Create directory if it doesn't exist

print("="*80)
print("💾 SAVING PREPROCESSED DATA")
print("="*80)

# --- 1. Save cleaned dataset (before scaling) ---
clean_path = os.path.join(PROCESSED_DIR, 'phishing_email_cleaned.csv')
df_clean.to_csv(clean_path, index=False)
print(f"   ✅ Cleaned dataset       → {clean_path}")

# --- 2. Save scaled Train set ---
train_path = os.path.join(PROCESSED_DIR, 'X_train_scaled.csv')
X_train_scaled.to_csv(train_path, index=False)
print(f"   ✅ X_train_scaled         → {train_path}")

# --- 3. Save scaled Test set ---
test_path = os.path.join(PROCESSED_DIR, 'X_test_scaled.csv')
X_test_scaled.to_csv(test_path, index=False)
print(f"   ✅ X_test_scaled          → {test_path}")

# --- 4. Save Train and Test labels ---
y_train.to_csv(os.path.join(PROCESSED_DIR, 'y_train.csv'), index=False, header=['label'])
print(f"   ✅ y_train                → {os.path.join(PROCESSED_DIR, 'y_train.csv')}")

y_test.to_csv(os.path.join(PROCESSED_DIR, 'y_test.csv'), index=False, header=['label'])
print(f"   ✅ y_test                 → {os.path.join(PROCESSED_DIR, 'y_test.csv')}")

print(f"\n🎉 Complete! All data has been saved to: {PROCESSED_DIR}")

### 4.8 Preprocessing Pipeline Summary

In [ ]:
# ============================================================
# 4.8 — Complete preprocessing pipeline summary table
# ============================================================

print("\n" + "="*80)
print("📋 COMPLETE DATA PREPROCESSING PIPELINE SUMMARY")
print("="*80)

pipeline_summary = [
    ['1. Load raw data', f'{df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns'],
    ['2. Feature Engineering (text → numeric)', f'Extracted {len(numeric_cols)} numerical features from text'],
    ['3. Handle Missing Values', 'Empty string (text) / median (numeric) / drop (target)'],
    ['4. Remove Duplicates', f'Removed {n_dups:,} duplicate rows'],
    ['5. Handle Outliers', f'IQR × 1.5 Winsorize on {len(numeric_features)} columns'],
    ['6. Label Encoding', f'{label_mapping}'],
    ['7. Train/Test Split', f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,} (80/20, stratified)'],
    ['8. Feature Scaling', 'StandardScaler (fit on Train → transform Test)'],
    ['9. Final dataset', f'{df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns'],
]

summary_df = pd.DataFrame(pipeline_summary, columns=['Step', 'Details'])
display(summary_df)

---

**Data Scientist's Notes:**

> _[Analysis notes after running Section 4 — e.g.: Total rows removed, whether the missing/outlier treatment strategy is sound, whether scaling results look normal (mean ≈ 0, std ≈ 1), any adjustments needed before feeding into the model, ...]_

---

---

## 📝 Final Summary

### Data Analysis Key Findings

- **Dataset**: ~18,650 emails (after removing original index), with 2 main columns: `Email Text` and `Email Type`
- **Class Balance**: Check the Safe/Phishing ratio after running — if imbalance > 3:1, consider applying SMOTE or class_weight
- **Feature Engineering**: Extracted 10 numerical features from text (char_count, word_count, uppercase_ratio, url_count, ...)
- **Correlation**: Check for multicollinearity between char_count ↔ word_count (likely high correlation > 0.9)
- **Outliers**: Handled via Winsorize (IQR clipping) instead of row removal — preserves dataset size
- **Data Leakage Prevention**: StandardScaler fitted only on Train, transformed on Test

### Insights & Next Steps

- **Next Step 1**: Perform advanced NLP Preprocessing (TF-IDF, Word Embeddings, DistilBERT tokenization) on the `Email Text` column to create feature vectors for Deep Learning models
- **Next Step 2**: Train a classification model (DistilBERT Fine-tuning) within the Federated Learning architecture

---

> **Notebook created following the CRISP-DM standard**  
> Author: Data Scientist | Project: FL Phishing Email Detection